In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append(str(Path.cwd().parent))
import src.config as cfg

# Display settings for operations data
pd.set_option('display.max_columns', None)

In [2]:
# Load raw data using our config path
df = pd.read_csv(cfg.RAW_DATA_FILE)

print(f"Dataset Shape: {df.shape}")
print("-" * 40)
print("Missing Values:")
miss = df.isnull().sum()[df.isnull().sum() > 0]
if miss.empty:
    print("No missing data found")
else:
    print(miss) # Only print if there are nulls
print("-" * 40)
# Check for Duplicates
duplicate_udi = df.duplicated(subset=['UDI']).sum()
print(f"Duplicate UDI count: {duplicate_udi}")

Dataset Shape: (10000, 14)
----------------------------------------
Missing Values:
No missing data found
----------------------------------------
Duplicate UDI count: 0


In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
# Check if Air Temperature is ever greater than Process Temperature
# In industrial machining, the process/tool should be hotter than the ambient air.
temp_violation = df[df['Air temperature [K]'] > df['Process temperature [K]']]
print(f"Temperature boundary violations (Air > Process): {len(temp_violation)}")

# Summary statistics for sensor ranges to check for impossible 0s or negatives
sensor_summary = df[cfg.NUMERIC_FEATURES].describe().T[['min', 'max', 'mean']]
print("\nSensor Physical Ranges:")
display(sensor_summary)

Temperature boundary violations (Air > Process): 0

Sensor Physical Ranges:


,min,max,mean
Air temperature [K],295.3,304.5,300.00493
Process temperature [K],305.7,313.8,310.00556
Rotational speed [rpm],1168.0,2886.0,1538.77610
Torque [Nm],3.8,76.6,39.98691
Tool wear [min],0.0,253.0,107.95100


In [5]:
# Calculate how many specific failure modes triggered per row
df['Total_Specific_Failures'] = df[cfg.FAILURE_MODES].sum(axis=1)

# Check 1: Did a specific failure trigger, but the master flag didn't?
missed_master_flag = df[(df['Total_Specific_Failures'] > 0) & (df[cfg.TARGET_COL] == 0)]
print(f"Specific failure triggered but Master Flag is 0: {len(missed_master_flag)}")

# Check 2: Did the master flag trigger, but no specific failure was logged?
# (This happens in real life - unknown failures)
unknown_failures = df[(df['Total_Specific_Failures'] == 0) & (df[cfg.TARGET_COL] == 1)]
print(f"Master Flag triggered but NO specific failure logged: {len(unknown_failures)}")

# Check 3: Cascading failures (More than one failure mode at the same time)
multiple_failures = df[df['Total_Specific_Failures'] > 1]
print(f"Machines with multiple simultaneous failure modes: {len(multiple_failures)}")

Specific failure triggered but Master Flag is 0: 18
Master Flag triggered but NO specific failure logged: 9
Machines with multiple simultaneous failure modes: 24


In [6]:
# Since the raw data is synthetic and likely clean, we will just drop the 
# 'Total_Specific_Failures' column we used for checking, and save to interim.
df_clean = df.drop(columns=['Total_Specific_Failures'])
df_clean.to_csv(cfg.INTERIM_DATA_DIR / "01_validated_data.csv", index=False)
print("Validated data saved to 02_interim.")

Validated data saved to 02_interim.
